# Customer Churn Prediction & Retention ROI

## Project Overview
This notebook converts the original churn-prediction project into a complete, presentation-ready machine learning workflow.

**Goals**
1. Understand customer churn patterns through EDA.
2. Clean and preprocess customer data.
3. Train an interpretable **Logistic Regression** model.
4. Train a **Gradient Boosting** benchmark model.
5. Evaluate both models using classification metrics, ROC-AUC and lift.
6. Explain individual churn predictions using linear-model SHAP-style contributions.
7. Build a simple **uplift + ROI simulator** to identify customers worth targeting.

> **Note:** The dataset is synthetic SaaS customer data supplied with the project. The retention-campaign fields allow us to demonstrate uplift modeling and expected-value targeting.


## 1. Import Required Libraries
We use:
- **Pandas / NumPy** for data manipulation.
- **Matplotlib / Seaborn** for visualization.
- **Scikit-learn** for preprocessing, modeling and evaluation.
- **Plotly** for interactive visualizations where useful.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve, precision_recall_curve
)

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
DATA_PATH = "../data/customers.csv"

df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df.shape}")
df.head()


## 2. Understand the Dataset
The dataset contains customer demographics/behavior, billing information, satisfaction, contract details and the final `churned` target.

Important columns include:
- `tenure_months` — customer tenure.
- `monthly_charges` — monthly billing amount.
- `login_freq_per_month` — engagement.
- `feature_adoption_score` — product adoption.
- `support_tickets_90d` — recent support activity.
- `satisfaction_score` — satisfaction from 0–10, with some missing responses.
- `contract_type` and `plan_tier` — categorical business attributes.
- `churned` — target: **1 = churned, 0 = stayed**.
- `was_offered_discount` and `stayed_after_offer` — historical campaign information for uplift analysis.


In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nFirst 5 rows:")
display(df.head())

print("\nStatistical summary:")
display(df.describe(include="all").T)


## 3. Data Quality Check
Before modeling, check duplicates, missing values and target balance.

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
missing_table = pd.DataFrame({"Missing Values": missing, "Missing %": missing_pct})
display(missing_table[missing_table["Missing Values"] > 0])

print("Target distribution:")
display(df["churned"].value_counts().rename(index={0:"Stayed", 1:"Churned"}))


In [ ]:
plt.figure(figsize=(7, 5))
sns.countplot(data=df, x="churned")
plt.title("Customer Churn Distribution")
plt.xlabel("Churned (0 = No, 1 = Yes)")
plt.ylabel("Number of Customers")
plt.show()

churn_rate = df["churned"].mean()
print(f"Overall churn rate: {churn_rate:.2%}")


## 4. Exploratory Data Analysis (EDA)

EDA helps identify which customer characteristics appear associated with churn. These relationships are exploratory and should not automatically be interpreted as causal.


In [ ]:
numeric_cols = [
    "tenure_months", "monthly_charges", "login_freq_per_month",
    "feature_adoption_score", "support_tickets_90d",
    "unresolved_ticket", "late_payments_90d",
    "satisfaction_score", "auto_renew"
]

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    sns.histplot(data=df, x=col, hue="churned", kde=True, element="step", ax=axes[i])
    axes[i].set_title(f"{col} Distribution by Churn")

plt.tight_layout()
plt.show()


In [ ]:
# Churn rate by important categorical variables

for col in ["contract_type", "plan_tier", "auto_renew", "unresolved_ticket"]:
    summary = df.groupby(col)["churned"].mean().sort_values(ascending=False) * 100
    print(f"\nChurn rate by {col}:")
    display(summary.round(2).to_frame("Churn Rate (%)"))

    plt.figure(figsize=(8, 5))
    sns.barplot(x=summary.index.astype(str), y=summary.values)
    plt.title(f"Churn Rate by {col}")
    plt.xlabel(col)
    plt.ylabel("Churn Rate (%)")
    plt.xticks(rotation=20)
    plt.show()


In [ ]:
# Correlation heatmap for numeric variables
plt.figure(figsize=(12, 8))
corr = df[numeric_cols + ["churned"]].corr(numeric_only=True)
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.show()


In [ ]:
# Compare average behavior for churned vs retained customers
comparison = df.groupby("churned")[numeric_cols].mean().T
comparison.columns = ["Stayed", "Churned"]
display(comparison.round(3))

comparison.plot(kind="bar", figsize=(14, 6))
plt.title("Average Customer Characteristics: Stayed vs Churned")
plt.ylabel("Average Value")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 5. Feature Engineering & Preprocessing

### Missing satisfaction values
About 30% of satisfaction surveys may be missing. We:
1. Create `satisfaction_missing` to preserve the fact that the customer did not respond.
2. Replace the missing score with the training-data median.

### Categorical variables
Contract and plan type are converted to one-hot encoded columns.

### Scaling
Logistic Regression benefits from standardized numeric features, so the scaler is fitted **only on the training data** to prevent data leakage.


In [ ]:
NUMERIC_FEATURES = [
    "tenure_months", "monthly_charges", "login_freq_per_month",
    "feature_adoption_score", "support_tickets_90d",
    "unresolved_ticket", "late_payments_90d",
    "satisfaction_score", "auto_renew"
]
CATEGORICAL_FEATURES = ["contract_type", "plan_tier"]

work_df = df.copy()
work_df["satisfaction_missing"] = work_df["satisfaction_score"].isna().astype(int)
median_satisfaction = work_df["satisfaction_score"].median()
work_df["satisfaction_score"] = work_df["satisfaction_score"].fillna(median_satisfaction)

encoded = pd.get_dummies(
    work_df, columns=CATEGORICAL_FEATURES, drop_first=True
)

feature_cols = (
    NUMERIC_FEATURES
    + ["satisfaction_missing"]
    + [
        c for c in encoded.columns
        if c.startswith("contract_type_") or c.startswith("plan_tier_")
    ]
)

X = encoded[feature_cols].astype(float)
y = encoded["churned"].astype(int)

print("Median satisfaction used for imputation:", median_satisfaction)
print("Number of model features:", len(feature_cols))
print("Features:")
print(feature_cols)


## 6. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print(f"Training churn rate: {y_train.mean():.2%}")
print(f"Testing churn rate: {y_test.mean():.2%}")


## 7. Model 1 — Logistic Regression

Logistic Regression is selected as the primary interpretable model.

`class_weight="balanced"` gives additional weight to the minority class and avoids requiring SMOTE for this workflow.


In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

logreg = LogisticRegression(
    class_weight="balanced",
    max_iter=2000,
    random_state=RANDOM_STATE
)

logreg.fit(X_train_scaled, y_train)

lr_proba = logreg.predict_proba(X_test_scaled)[:, 1]
lr_pred = (lr_proba >= 0.50).astype(int)

print("Logistic Regression trained successfully.")


## 8. Model 2 — Gradient Boosting

Gradient Boosting is used as a higher-capacity benchmark. It can learn nonlinear relationships that a linear model may miss.

Because Gradient Boosting does not expose `class_weight`, we emulate class balancing using `sample_weight`.


In [ ]:
gbm = GradientBoostingClassifier(
    n_estimators=250,
    max_depth=3,
    learning_rate=0.05,
    random_state=RANDOM_STATE
)

positive_weight = (y_train == 0).sum() / (y_train == 1).sum()
sample_weight = np.where(y_train == 1, positive_weight, 1.0)

gbm.fit(X_train, y_train, sample_weight=sample_weight)

gbm_proba = gbm.predict_proba(X_test)[:, 1]
gbm_pred = (gbm_proba >= 0.50).astype(int)

print("Gradient Boosting trained successfully.")


## 9. Model Evaluation

In [ ]:
def evaluate_model(name, y_true, pred, proba):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, pred),
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Recall": recall_score(y_true, pred, zero_division=0),
        "F1": f1_score(y_true, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, proba)
    }

results = pd.DataFrame([
    evaluate_model("Logistic Regression", y_test, lr_pred, lr_proba),
    evaluate_model("Gradient Boosting", y_test, gbm_pred, gbm_proba)
])

display(results.round(4))


In [ ]:
# Detailed classification reports
print("LOGISTIC REGRESSION")
print(classification_report(y_test, lr_pred, target_names=["Stayed", "Churned"]))

print("\nGRADIENT BOOSTING")
print(classification_report(y_test, gbm_pred, target_names=["Stayed", "Churned"]))


In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, pred, title in [
    (axes[0], lr_pred, "Logistic Regression"),
    (axes[1], gbm_pred, "Gradient Boosting")
]:
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
    ax.set_title(f"Confusion Matrix — {title}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.show()


In [ ]:
# ROC curves
lr_fpr, lr_tpr, _ = roc_curve(y_test, lr_proba)
gbm_fpr, gbm_tpr, _ = roc_curve(y_test, gbm_proba)

plt.figure(figsize=(8, 6))
plt.plot(lr_fpr, lr_tpr, label=f"Logistic Regression (AUC={roc_auc_score(y_test, lr_proba):.3f})")
plt.plot(gbm_fpr, gbm_tpr, label=f"Gradient Boosting (AUC={roc_auc_score(y_test, gbm_proba):.3f})")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.show()


In [ ]:
# Precision-Recall curves
lr_precision, lr_recall, _ = precision_recall_curve(y_test, lr_proba)
gbm_precision, gbm_recall, _ = precision_recall_curve(y_test, gbm_proba)

plt.figure(figsize=(8, 6))
plt.plot(lr_recall, lr_precision, label="Logistic Regression")
plt.plot(gbm_recall, gbm_precision, label="Gradient Boosting")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve Comparison")
plt.legend()
plt.show()


## 10. Lift at Top K%

Lift answers a practical business question:

> If the retention team can contact only the highest-risk customers, how much better is that group than selecting customers randomly?

A lift of 2.0 at 10% means the top 10% risk-ranked group has approximately twice the churn rate of the overall test population.


In [ ]:
def lift_at_k(y_true, y_score, k_pct=0.10):
    n = len(y_true)
    k = max(1, int(n * k_pct))
    order = np.argsort(-np.asarray(y_score))
    top_k = np.asarray(y_true)[order[:k]]
    base_rate = np.mean(y_true)
    return float(np.mean(top_k) / base_rate) if base_rate > 0 else np.nan

lift_table = pd.DataFrame({
    "Model": ["Logistic Regression", "Gradient Boosting"],
    "Lift @ 10%": [
        lift_at_k(y_test, lr_proba, 0.10),
        lift_at_k(y_test, gbm_proba, 0.10)
    ],
    "Lift @ 20%": [
        lift_at_k(y_test, lr_proba, 0.20),
        lift_at_k(y_test, gbm_proba, 0.20)
    ]
})

display(lift_table.round(3))

lift_plot = lift_table.set_index("Model")
lift_plot.plot(kind="bar", figsize=(9, 5))
plt.title("Top-K Lift Comparison")
plt.ylabel("Lift")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 11. Feature Importance — Logistic Regression

For a standardized Logistic Regression model:
- A **positive coefficient** increases predicted churn risk.
- A **negative coefficient** decreases predicted churn risk.
- The larger the absolute coefficient, the stronger the feature's effect on the model's log-odds scale.

This gives a useful, interview-friendly explanation of why the model makes predictions.


In [ ]:
lr_importance = pd.DataFrame({
    "Feature": feature_cols,
    "Coefficient": logreg.coef_[0]
})
lr_importance["Absolute Importance"] = lr_importance["Coefficient"].abs()
lr_importance = lr_importance.sort_values("Absolute Importance", ascending=False)

display(lr_importance.round(4))

plt.figure(figsize=(10, 7))
top = lr_importance.head(15).sort_values("Coefficient")
sns.barplot(data=top, x="Coefficient", y="Feature")
plt.axvline(0, linestyle="--")
plt.title("Top Logistic Regression Coefficients")
plt.xlabel("Coefficient (standardized feature)")
plt.ylabel("Feature")
plt.show()


## 12. Feature Importance — Gradient Boosting

In [ ]:
gbm_importance = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": gbm.feature_importances_
}).sort_values("Importance", ascending=False)

display(gbm_importance.head(15).round(4))

plt.figure(figsize=(10, 7))
sns.barplot(
    data=gbm_importance.head(15).sort_values("Importance"),
    x="Importance",
    y="Feature"
)
plt.title("Top Gradient Boosting Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()


## 13. Customer-Level Explainability

For Logistic Regression, an exact additive decomposition of the model's **logit** can be written as:

`contribution_i = coefficient_i × standardized_feature_i`

The sum of all feature contributions plus the intercept equals the model's logit. Applying the sigmoid function converts that logit into churn probability.

This is the linear-model SHAP-style explanation used in the original project.


In [ ]:
# Save the exact values required for customer explanations
model_means = scaler.mean_
model_scales = scaler.scale_
model_coefs = logreg.coef_[0]
model_intercept = logreg.intercept_[0]

def build_feature_row(row):
    d = {
        "tenure_months": row["tenure_months"],
        "monthly_charges": row["monthly_charges"],
        "login_freq_per_month": row["login_freq_per_month"],
        "feature_adoption_score": row["feature_adoption_score"],
        "support_tickets_90d": row["support_tickets_90d"],
        "unresolved_ticket": row["unresolved_ticket"],
        "late_payments_90d": row["late_payments_90d"],
        "satisfaction_score": row["satisfaction_score"],
        "auto_renew": row["auto_renew"],
        "satisfaction_missing": int(pd.isna(row["satisfaction_score"])),
        "contract_type_Month-to-month": int(row["contract_type"] == "Month-to-month"),
        "contract_type_One year": int(row["contract_type"] == "One year"),
        "contract_type_Two year": int(row["contract_type"] == "Two year"),
        "plan_tier_Basic": int(row["plan_tier"] == "Basic"),
        "plan_tier_Enterprise": int(row["plan_tier"] == "Enterprise"),
        "plan_tier_Pro": int(row["plan_tier"] == "Pro"),
    }
    if pd.isna(d["satisfaction_score"]):
        d["satisfaction_score"] = median_satisfaction
    return np.array([d[f] for f in feature_cols], dtype=float)

def explain_customer(customer_id, top_k=8):
    matches = df[df["customer_id"] == customer_id]
    if matches.empty:
        raise ValueError(f"{customer_id} not found")
    row = matches.iloc[0]

    x = build_feature_row(row)
    x_scaled = (x - model_means) / model_scales
    contributions = model_coefs * x_scaled
    logit = contributions.sum() + model_intercept
    probability = 1 / (1 + np.exp(-logit))

    explanation = pd.DataFrame({
        "Feature": feature_cols,
        "Raw Value": x,
        "Contribution": contributions,
        "Direction": np.where(
            contributions > 0, "Increases risk", "Decreases risk"
        )
    })
    explanation["Absolute Contribution"] = explanation["Contribution"].abs()
    explanation = explanation.sort_values(
        "Absolute Contribution", ascending=False
    ).head(top_k)

    return probability, explanation

# Automatically choose an example customer
example_customer = df.iloc[0]["customer_id"]
prob, explanation = explain_customer(example_customer)

print(f"Customer: {example_customer}")
print(f"Predicted churn probability: {prob:.2%}")
display(explanation.round(4))


In [ ]:
plt.figure(figsize=(10, 6))
plot_data = explanation.sort_values("Contribution")
sns.barplot(data=plot_data, x="Contribution", y="Feature")
plt.axvline(0, linestyle="--")
plt.title(f"Top Churn Drivers — {example_customer}")
plt.xlabel("Contribution to churn log-odds")
plt.ylabel("Feature")
plt.show()


## 14. Score All Customers

We now use the trained Logistic Regression model to generate a churn probability for every customer. This creates a practical risk-ranking table for a retention team.


In [ ]:
X_all_scaled = scaler.transform(X)
df_scored = df.copy()
df_scored["churn_probability"] = logreg.predict_proba(X_all_scaled)[:, 1]
df_scored["risk_segment"] = pd.cut(
    df_scored["churn_probability"],
    bins=[-np.inf, 0.25, 0.50, 0.75, np.inf],
    labels=["Low", "Medium", "High", "Very High"]
)

display(
    df_scored[
        ["customer_id", "churn_probability", "risk_segment", "contract_type",
         "plan_tier", "monthly_charges", "tenure_months"]
    ].sort_values("churn_probability", ascending=False).head(20)
)


In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(df_scored["churn_probability"], bins=30, kde=True)
plt.title("Distribution of Predicted Churn Probability")
plt.xlabel("Predicted Churn Probability")
plt.ylabel("Number of Customers")
plt.show()

segment_counts = df_scored["risk_segment"].value_counts().reindex(
    ["Low", "Medium", "High", "Very High"]
)

plt.figure(figsize=(8, 5))
sns.barplot(x=segment_counts.index, y=segment_counts.values)
plt.title("Customers by Risk Segment")
plt.xlabel("Risk Segment")
plt.ylabel("Customers")
plt.show()


## 15. Uplift Modeling for Retention

Raw churn risk answers **who is likely to leave**.

Uplift modeling asks a more valuable question:

> **Who is likely to stay because we offer a retention discount?**

The dataset contains a historical randomized discount offer:
- `was_offered_discount = 1` → treatment group.
- `was_offered_discount = 0` → control group.

We train two Logistic Regression models:
- `P(stay | offered discount, X)`
- `P(stay | not offered discount, X)`

Then:

`uplift = P(stay | offer, X) - P(stay | no offer, X)`

Positive uplift means the offer is estimated to improve retention.


In [ ]:
UPLIFT_NUMERIC = [
    "tenure_months", "monthly_charges", "login_freq_per_month",
    "feature_adoption_score", "support_tickets_90d",
    "unresolved_ticket", "late_payments_90d", "auto_renew"
]
UPLIFT_CATEGORICAL = ["contract_type", "plan_tier"]

uplift_df = df.copy()
uplift_df["satisfaction_score"] = uplift_df["satisfaction_score"].fillna(
    uplift_df["satisfaction_score"].median()
)

uplift_encoded = pd.get_dummies(
    uplift_df, columns=UPLIFT_CATEGORICAL, drop_first=True
)

uplift_features = (
    UPLIFT_NUMERIC + ["satisfaction_score"] +
    [c for c in uplift_encoded.columns
     if c.startswith("contract_type_") or c.startswith("plan_tier_")]
)

X_uplift = uplift_encoded[uplift_features].astype(float)

treated = uplift_df["was_offered_discount"] == 1
control = uplift_df["was_offered_discount"] == 0

y_stay_treated = 1 - uplift_df.loc[treated, "churned"]
y_stay_control = 1 - uplift_df.loc[control, "churned"]

uplift_scaler = StandardScaler().fit(X_uplift)

X_treated = uplift_scaler.transform(X_uplift.loc[treated])
X_control = uplift_scaler.transform(X_uplift.loc[control])
X_all_uplift = uplift_scaler.transform(X_uplift)

model_treated = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
model_control = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)

model_treated.fit(X_treated, y_stay_treated)
model_control.fit(X_control, y_stay_control)

p_stay_treated = model_treated.predict_proba(X_all_uplift)[:, 1]
p_stay_control = model_control.predict_proba(X_all_uplift)[:, 1]

uplift_df["p_stay_if_offered"] = p_stay_treated
uplift_df["p_stay_if_not_offered"] = p_stay_control
uplift_df["uplift"] = p_stay_treated - p_stay_control

display(
    uplift_df[
        ["customer_id", "contract_type", "plan_tier",
         "p_stay_if_offered", "p_stay_if_not_offered", "uplift"]
    ].sort_values("uplift", ascending=False).head(20).round(4)
)


In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(uplift_df["uplift"], bins=40, kde=True)
plt.axvline(0, linestyle="--")
plt.title("Distribution of Estimated Retention Uplift")
plt.xlabel("Estimated Uplift")
plt.ylabel("Customers")
plt.show()

plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=uplift_df.sample(min(1500, len(uplift_df)), random_state=RANDOM_STATE),
    x="churned",
    y="uplift",
    hue="contract_type"
)
plt.title("Estimated Uplift by Observed Churn Outcome")
plt.xlabel("Observed Churn")
plt.ylabel("Estimated Uplift")
plt.show()


## 16. ROI Simulation

Assume:
- Discount/intervention cost = **$50**
- Customer lifetime value (LTV) is approximated from monthly charges, margin and expected remaining months.

Expected value:

`EV = uplift × LTV − intervention_cost`

We recommend an offer only when `EV > 0`.


In [ ]:
def compute_ltv(data, monthly_margin_pct=0.55, horizon_months=24):
    remaining = np.clip(
        horizon_months - (data["tenure_months"] % horizon_months),
        3,
        horizon_months
    )
    return (
        data["monthly_charges"] *
        monthly_margin_pct *
        remaining
    )

discount_cost = 50.0

uplift_df["ltv_estimate"] = compute_ltv(uplift_df)
uplift_df["expected_value_of_offer"] = (
    uplift_df["uplift"] * uplift_df["ltv_estimate"] - discount_cost
)
uplift_df["recommend_offer"] = uplift_df["expected_value_of_offer"] > 0

roi_cols = [
    "customer_id", "churned", "contract_type", "plan_tier",
    "uplift", "ltv_estimate", "expected_value_of_offer",
    "recommend_offer"
]

display(
    uplift_df[roi_cols]
    .sort_values("expected_value_of_offer", ascending=False)
    .head(25)
    .round(3)
)

print(
    f"Recommended offers: {uplift_df['recommend_offer'].sum()} "
    f"out of {len(uplift_df)} customers "
    f"({uplift_df['recommend_offer'].mean():.2%})"
)


In [ ]:
# Segment customers into practical treatment groups
uplift_df["segment"] = np.select(
    [
        (uplift_df["uplift"] > 0.05) & (uplift_df["churned"] == 1),
        (uplift_df["uplift"] > 0.05) & (uplift_df["churned"] == 0),
        (uplift_df["uplift"] <= 0.05) & (uplift_df["churned"] == 1),
        (uplift_df["uplift"] <= 0.05) & (uplift_df["churned"] == 0),
    ],
    [
        "Persuadable / High Value",
        "Persuadable / Currently Safe",
        "Low-Uplift / Churn Risk",
        "Low-Uplift / Safe"
    ],
    default="Other"
)

segment_summary = uplift_df.groupby("segment").agg(
    customers=("customer_id", "count"),
    avg_uplift=("uplift", "mean"),
    avg_ltv=("ltv_estimate", "mean"),
    avg_expected_value=("expected_value_of_offer", "mean")
).sort_values("avg_expected_value", ascending=False)

display(segment_summary.round(3))

plt.figure(figsize=(10, 5))
sns.barplot(
    data=segment_summary.reset_index(),
    x="segment",
    y="customers"
)
plt.title("Retention Strategy Segments")
plt.xlabel("Segment")
plt.ylabel("Number of Customers")
plt.xticks(rotation=25, ha="right")
plt.show()


## 17. Compare Risk-Based vs Uplift-Based Targeting

A major business insight is that the customer with the highest churn probability is **not necessarily** the customer who should receive a discount.

- **Risk model:** prioritizes probability of churn.
- **Uplift model:** prioritizes estimated incremental benefit from intervention.
- **ROI model:** adds customer value and intervention cost.

This creates a decision pipeline:

**Predict → Explain → Estimate intervention effect → Calculate ROI → Target**


In [ ]:
# Top 20 customers by churn risk
top_risk = df_scored.nlargest(20, "churn_probability")[
    ["customer_id", "churn_probability", "contract_type", "plan_tier"]
]

# Top 20 customers by expected offer value
top_roi = uplift_df.nlargest(20, "expected_value_of_offer")[
    ["customer_id", "uplift", "ltv_estimate", "expected_value_of_offer",
     "contract_type", "plan_tier"]
]

print("TOP 20 BY CHURN RISK")
display(top_risk.round(4))

print("\nTOP 20 BY RETENTION OFFER ROI")
display(top_roi.round(4))


## 18. Export the Final Scored Dataset

The final table combines customer information with:
- churn probability,
- risk segment,
- treatment response estimates,
- uplift,
- estimated LTV,
- expected value,
- retention recommendation.

This is the table that could feed a CRM/retention dashboard in a real deployment.


In [ ]:
final_scored = uplift_df.copy()
final_scored["churn_probability"] = df_scored["churn_probability"].values
final_scored["risk_segment"] = df_scored["risk_segment"].values

final_output = final_scored.sort_values(
    ["recommend_offer", "expected_value_of_offer"],
    ascending=[False, False]
)

OUTPUT_PATH = "../outputs/churn_analysis_scored.csv"
os.makedirs("../outputs", exist_ok=True)
final_output.to_csv(OUTPUT_PATH, index=False)

print(f"Saved: {OUTPUT_PATH}")
display(final_output.head(10))


# Final Project Summary

### What we implemented
- Complete data loading and quality checks.
- Missing-value handling and categorical encoding.
- Exploratory analysis with multiple business-focused graphs.
- Logistic Regression for interpretable churn prediction.
- Gradient Boosting as a higher-capacity benchmark.
- Accuracy, precision, recall, F1, ROC-AUC and confusion-matrix evaluation.
- ROC and Precision-Recall curves.
- Top-K lift analysis for campaign prioritization.
- Global feature importance.
- Customer-level churn explanations using exact linear-model contribution decomposition.
- Uplift modeling using the historical discount experiment.
- LTV and expected-value calculations.
- Retention campaign recommendations based on positive expected ROI.
- Export of the final customer scoring table.

### Key takeaway
The strongest version of a churn system should not stop at **"Who will churn?"**. A business-ready system should answer three questions:

1. **Who is at risk?**
2. **Why are they at risk?**
3. **Who is actually worth intervening on?**

That final step connects machine learning predictions to a measurable business decision.
